# 01 - 路由分类器训练（BERT 多标签）

**目标**：训练一个 BERT 模型，给定用户问题，预测需要哪些角色（physician / pharmacist / radiologist）参与会诊。

**模型**：`bert-base-chinese` (110M 参数)

**数据**：`datasets/data/router_train.jsonl`（运行 `gen_router_data.py` 生成）

**预期**：
- 训练时间：约 20 分钟（5070 Ti 笔记本）
- 显存：2-3 GB
- 准确率：> 85%（每个标签的 F1）

In [1]:
import os
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
print('HF mirror 已设置')

HF mirror 已设置


## Step 1: 环境检查

In [2]:
import torch
import transformers
import sys

print(f'Python: {sys.version}')
print(f'PyTorch: {torch.__version__}')
print(f'Transformers: {transformers.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Python: 3.11.15 | packaged by Anaconda, Inc. | (main, Mar 11 2026, 17:12:15) [MSC v.1942 64 bit (AMD64)]
PyTorch: 2.7.0+cu128
Transformers: 5.6.2
CUDA available: True
GPU: NVIDIA GeForce RTX 5070 Ti Laptop GPU
VRAM: 12.8 GB


## Step 2: 配置参数（你可以根据显存大小调整）

In [3]:
from pathlib import Path

# ===== 可配置参数 =====
MODEL_NAME = 'bert-base-chinese'  # 也可以换成 hfl/chinese-roberta-wwm-ext
DATA_PATH = Path('../datasets/data/router_train.jsonl')
OUTPUT_DIR = Path('../models/router_bert')

MAX_LENGTH = 128
BATCH_SIZE = 16    # 显存不够改成 8
LEARNING_RATE = 2e-5
NUM_EPOCHS = 5
VAL_RATIO = 0.1
TEST_RATIO = 0.1
SEED = 42

LABELS = ['physician', 'pharmacist', 'radiologist']
NUM_LABELS = len(LABELS)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'输出目录: {OUTPUT_DIR.resolve()}')

输出目录: D:\NLP_Project\miniOpenClaw\backend\eval\models\router_bert


## Step 3: 加载数据

In [4]:
import json
import random

random.seed(SEED)

records = []
with open(DATA_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

print(f'总数据量: {len(records)}')
print(f'前 3 条样例:')
for r in records[:3]:
    print(json.dumps(r, ensure_ascii=False))

总数据量: 1002
前 3 条样例:
{"query": "我最近总是肚子疼和拉肚子，不知道是什么原因？", "labels": {"physician": 1, "pharmacist": 0, "radiologist": 0}, "scene": "纯诊断问题（只需要主治）"}
{"query": "孩子发烧三天了，温度反反复复，该不会是肺炎吧？", "labels": {"physician": 1, "pharmacist": 0, "radiologist": 0}, "scene": "纯诊断问题（只需要主治）"}
{"query": "老人最近总是头晕，站不稳，是怎么回事？", "labels": {"physician": 1, "pharmacist": 0, "radiologist": 0}, "scene": "纯诊断问题（只需要主治）"}


In [5]:
# 标签分布统计
from collections import Counter

label_counts = Counter()
for r in records:
    pattern = tuple(r['labels'][label] for label in LABELS)
    label_counts[pattern] += 1

print('标签分布 (physician, pharmacist, radiologist) -> 数量:')
for pattern, count in sorted(label_counts.items(), key=lambda x: -x[1]):
    print(f'  {pattern}: {count}')

标签分布 (physician, pharmacist, radiologist) -> 数量:
  (1, 0, 1): 335
  (1, 1, 0): 332
  (1, 0, 0): 175
  (1, 1, 1): 160


In [6]:
# 划分数据集
random.shuffle(records)
n = len(records)
n_test = int(n * TEST_RATIO)
n_val = int(n * VAL_RATIO)
n_train = n - n_test - n_val

train_records = records[:n_train]
val_records = records[n_train:n_train + n_val]
test_records = records[n_train + n_val:]

print(f'训练: {len(train_records)}, 验证: {len(val_records)}, 测试: {len(test_records)}')

训练: 802, 验证: 100, 测试: 100


## Step 4: 准备 Tokenizer 和 Dataset

In [7]:
from transformers import AutoTokenizer
from torch.utils.data import Dataset

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class RouterDataset(Dataset):
    def __init__(self, records, tokenizer, max_length=128):
        self.records = records
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.records)
    
    def __getitem__(self, idx):
        r = self.records[idx]
        encoding = self.tokenizer(
            r['query'],
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        labels = torch.tensor([r['labels'][k] for k in LABELS], dtype=torch.float)
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': labels
        }

train_ds = RouterDataset(train_records, tokenizer, MAX_LENGTH)
val_ds = RouterDataset(val_records, tokenizer, MAX_LENGTH)
test_ds = RouterDataset(test_records, tokenizer, MAX_LENGTH)

print(f'数据集长度: train={len(train_ds)}, val={len(val_ds)}, test={len(test_ds)}')
print(f'第 1 条 input_ids 形状: {train_ds[0]["input_ids"].shape}')
print(f'第 1 条 labels: {train_ds[0]["labels"]}')

数据集长度: train=802, val=100, test=100
第 1 条 input_ids 形状: torch.Size([128])
第 1 条 labels: tensor([1., 0., 1.])


## Step 5: 加载模型（多标签分类头）

In [8]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    problem_type='multi_label_classification'  # 关键：多标签
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f'模型: {MODEL_NAME}')
print(f'参数量: {n_params / 1e6:.1f}M')
print(f'设备: {device}')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-chinese
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


模型: bert-base-chinese
参数量: 102.3M
设备: cuda


## Step 6: 训练（使用 Trainer API）

In [9]:
import os
os.environ['WANDB_DISABLED'] = 'true'  # 不想用 wandb 就保留这行；用就注释掉

from transformers import TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import f1_score, accuracy_score, classification_report

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))  # sigmoid
    preds = (probs > 0.5).astype(int)
    
    metrics = {
        'subset_accuracy': accuracy_score(labels, preds),  # 完全一致的比例
        'micro_f1': f1_score(labels, preds, average='micro', zero_division=0),
        'macro_f1': f1_score(labels, preds, average='macro', zero_division=0),
    }
    # 每个标签的 F1
    for i, label_name in enumerate(LABELS):
        metrics[f'f1_{label_name}'] = f1_score(labels[:, i], preds[:, i], zero_division=0)
    return metrics

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / 'checkpoints'),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    greater_is_better=True,
    logging_steps=20,
    seed=SEED,
    fp16=torch.cuda.is_available(),
    report_to='none',  # 不用 wandb
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,    # ← 这里
    compute_metrics=compute_metrics,
)

print('开始训练...')
trainer.train()

开始训练...


Epoch,Training Loss,Validation Loss,Subset Accuracy,Micro F1,Macro F1,F1 Physician,F1 Pharmacist,F1 Radiologist
1,0.241645,0.104814,0.970000,0.992366,0.989541,1.000000,0.978723,0.989899
2,0.041659,0.042530,0.980000,0.994924,0.992908,1.000000,0.978723,1.000000
3,0.023369,0.030189,0.990000,0.997455,0.996416,1.000000,0.989247,1.000000
4,0.016590,0.024831,0.990000,0.997455,0.996416,1.000000,0.989247,1.000000
5,0.014794,0.025369,0.990000,0.997455,0.996416,1.000000,0.989247,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=255, training_loss=0.08668250833071914, metrics={'train_runtime': 34.5527, 'train_samples_per_second': 116.055, 'train_steps_per_second': 7.38, 'total_flos': 263771201272320.0, 'train_loss': 0.08668250833071914, 'epoch': 5.0})

## Step 7: 测试集评估

In [10]:
test_results = trainer.evaluate(test_ds)
print('\n=== 测试集评估 ===')
for k, v in test_results.items():
    if k.startswith('eval_'):
        print(f'{k}: {v:.4f}')

Training Loss,Validation Loss,Epoch,Subset Accuracy,Micro F1,Macro F1,F1 Physician,F1 Pharmacist,F1 Radiologist
0.014794,0.015809,5,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000



=== 测试集评估 ===
eval_loss: 0.0158
eval_subset_accuracy: 1.0000
eval_micro_f1: 1.0000
eval_macro_f1: 1.0000
eval_f1_physician: 1.0000
eval_f1_pharmacist: 1.0000
eval_f1_radiologist: 1.0000


In [11]:
# 详细分类报告
test_preds = trainer.predict(test_ds)
logits = test_preds.predictions
labels_arr = test_preds.label_ids

probs = 1 / (1 + np.exp(-logits))
preds = (probs > 0.5).astype(int)

print('\n=== 各角色分类报告 ===')
for i, label_name in enumerate(LABELS):
    print(f'\n[{label_name}]')
    print(classification_report(
        labels_arr[:, i],
        preds[:, i],
        labels=[0, 1],                                    # ← 加这行
        target_names=[f'no_{label_name}', label_name],
        zero_division=0
    ))


=== 各角色分类报告 ===

[physician]
              precision    recall  f1-score   support

no_physician       0.00      0.00      0.00         0
   physician       1.00      1.00      1.00       100

    accuracy                           1.00       100
   macro avg       0.50      0.50      0.50       100
weighted avg       1.00      1.00      1.00       100


[pharmacist]
               precision    recall  f1-score   support

no_pharmacist       1.00      1.00      1.00        54
   pharmacist       1.00      1.00      1.00        46

     accuracy                           1.00       100
    macro avg       1.00      1.00      1.00       100
 weighted avg       1.00      1.00      1.00       100


[radiologist]
                precision    recall  f1-score   support

no_radiologist       1.00      1.00      1.00        47
   radiologist       1.00      1.00      1.00        53

      accuracy                           1.00       100
     macro avg       1.00      1.00      1.00       100

## Step 8: 保存模型

In [12]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# 保存 label 元信息
import json
with open(OUTPUT_DIR / 'label_config.json', 'w', encoding='utf-8') as f:
    json.dump({
        'labels': LABELS,
        'threshold': 0.5,
        'model_name': MODEL_NAME,
        'max_length': MAX_LENGTH,
        'test_metrics': {k: float(v) for k, v in test_results.items() if isinstance(v, (int, float))}
    }, f, ensure_ascii=False, indent=2)

print(f'模型已保存到: {OUTPUT_DIR.resolve()}')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

模型已保存到: D:\NLP_Project\miniOpenClaw\backend\eval\models\router_bert


## Step 9: 推理样例（验证模型工作正常）

In [13]:
model.eval()

test_queries = [
    '我最近头痛三天，吃什么药好？',
    '体检发现 5mm 肺结节，要不要做 CT 复查？',
    '孕妇感冒能用布洛芬吗？',
    '老人腰痛拍 X 光还是做 MRI？',
    '高血压患者吃阿司匹林时需要做什么影像检查吗？',
    '小孩发烧 39 度持续两天怎么办？',
]

with torch.no_grad():
    for q in test_queries:
        enc = tokenizer(q, return_tensors='pt', truncation=True, padding='max_length', max_length=MAX_LENGTH).to(device)
        logits = model(**enc).logits
        probs = torch.sigmoid(logits).cpu().numpy()[0]
        
        result = {LABELS[i]: f'{probs[i]:.2f}' for i in range(len(LABELS))}
        roles_needed = [LABELS[i] for i in range(len(LABELS)) if probs[i] > 0.5]
        print(f'\n问: {q}')
        print(f'  概率: {result}')
        print(f'  需要会诊: {" + ".join(roles_needed)}')


问: 我最近头痛三天，吃什么药好？
  概率: {'physician': '0.99', 'pharmacist': '0.98', 'radiologist': '0.01'}
  需要会诊: physician + pharmacist

问: 体检发现 5mm 肺结节，要不要做 CT 复查？
  概率: {'physician': '0.98', 'pharmacist': '0.01', 'radiologist': '0.99'}
  需要会诊: physician + radiologist

问: 孕妇感冒能用布洛芬吗？
  概率: {'physician': '0.99', 'pharmacist': '0.99', 'radiologist': '0.01'}
  需要会诊: physician + pharmacist

问: 老人腰痛拍 X 光还是做 MRI？
  概率: {'physician': '0.98', 'pharmacist': '0.02', 'radiologist': '0.99'}
  需要会诊: physician + radiologist

问: 高血压患者吃阿司匹林时需要做什么影像检查吗？
  概率: {'physician': '0.99', 'pharmacist': '0.99', 'radiologist': '0.25'}
  需要会诊: physician + pharmacist

问: 小孩发烧 39 度持续两天怎么办？
  概率: {'physician': '0.99', 'pharmacist': '0.03', 'radiologist': '0.02'}
  需要会诊: physician


## ✅ 完成

训练好的模型在：`backend/eval/models/router_bert/`

**下一步**：
1. 跑 `02_train_guardian.ipynb`
2. 或者先把这个模型集成到后端：`backend/eval/inference/load_router.py`